# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varshitha922/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Feature vector

I selected three features: `gsc_impressions`, `gsc_clicks`, and `gsc_avg_position`.

- **Impressions** show how many times a page appeared in Google Search.
- **Clicks** show how many times users clicked the page.
- **Average position** shows the page's average position in search results.

I handle missing numeric values by filling them with the median of the sample. I selected these features because they describe search performance and may help identify pages for review. They should be measured before the prediction point to avoid using future information.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb

import duckdb
import pandas as pd
from google.colab import userdata

token = userdata.get("HF_TOKEN")
con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '" + token + "')"
)

march = """read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)"""

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

# Load a consistent sample of March 2026 records
data = con.sql(f"""
    SELECT
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM {march}
    ORDER BY report_date, client_hash_id, content_hash_id
    LIMIT 10000
""").df()

# Build the feature vector
X = data[feature_columns].copy()

# Convert values to numeric
for col in feature_columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

# Fill missing values with the median
X = X.fillna(X.median())

print("Feature vector shape:", X.shape)
print("Features:", list(X.columns))
print("Missing values after filling:")
print(X.isna().sum())
print(X.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector shape: (10000, 3)
Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']
Missing values after filling:
gsc_impressions     0
gsc_clicks          0
gsc_avg_position    0
dtype: int64
   gsc_impressions  gsc_clicks  gsc_avg_position
0                0           0              10.0
1                0           0              10.0
2                0           0              10.0
3                0           0              10.0
4                0           0              10.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

| Feature | Meaning | Missing value handling | Categorical? | Available before prediction? |
|---|---|---|---|---|
| `gsc_impressions` | Number of times a page appeared in Google Search. | Missing values are filled with the median. | No, numeric. | Only if measured before the prediction point. |
| `gsc_clicks` | Number of clicks received from Google Search. | Missing values are filled with the median. | No, numeric. | Only if measured before the prediction point. |
| `gsc_avg_position` | Average position of a page in search results. | Missing values are filled with the median. | No, numeric. | Only if measured before the prediction point. |

These features describe past search performance. Their measurement period must end before the prediction point. The prediction date and feature window must be clearly defined.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2: Feature verification

print("Feature notes verification:")

print(data[feature_columns].dtypes)

print("\nMissing values before filling:")
print(data[feature_columns].isna().sum())

print("\nMissing values after filling:")
print(X[feature_columns].isna().sum())

print("\nFeature availability check:")
print("The selected features are historical Google Search metrics.")
print("They must be measured before the prediction point.")

Feature notes verification:
gsc_impressions       int64
gsc_clicks            int64
gsc_avg_position    float64
dtype: object

Missing values before filling:
gsc_impressions        0
gsc_clicks             0
gsc_avg_position    6751
dtype: int64

Missing values after filling:
gsc_impressions     0
gsc_clicks          0
gsc_avg_position    0
dtype: int64

Feature availability check:
The selected features are historical Google Search metrics.
They must be measured before the prediction point.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage hunt

I checked the selected features for possible data leakage. The selected features are `gsc_impressions`, `gsc_clicks`, and `gsc_avg_position`.

I did not include future-period fields, label-derived columns, or product flags. These could give the model information that would not be available at prediction time.

The code checks whether the selected feature names contain obvious future-related or label-related terms. This check does not prove that the values are free from leakage. The measurement period must also end before the prediction point.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: Leakage check

# Check that only the selected historical features are used
print("Selected features:")
print(list(X.columns))

# Check for suspicious future or label-related column names
suspicious_words = ["future", "label", "target", "outcome", "next", "product"]

suspicious_columns = [
    col for col in X.columns
    if any(word in col.lower() for word in suspicious_words)
]

print("\nPossible leakage columns:")
print(suspicious_columns)

# Confirm that no suspicious columns are included
assert len(suspicious_columns) == 0, "Possible leakage column found!"

print("\nLeakage check passed for the selected feature columns.")

Selected features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']

Possible leakage columns:
[]

Leakage check passed for the selected feature columns.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded fields and reasons

Based on the data contract, I excluded the following field groups from the feature vector:

- **Future-period fields:** Not used because they may contain information from after the prediction point and cause data leakage.
- **Label-derived columns:** Not used because they may reveal the outcome the model is trying to predict.
- **Product flags:** Not used because their availability at prediction time is uncertain.
- **Client names, URLs, and private queries:** Not used to protect privacy and avoid using identifying information.
- **Context fields such as `client_hash_id` and `content_hash_id`:** Not used as model features because they identify records rather than describe search performance.


In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Verify included and excluded fields

excluded_fields = {
    "Future-period fields": "Not used because they may contain information from after prediction time.",
    "Label-derived columns": "Not used because they may reveal the outcome.",
    "Product flags": "Not used because their availability at prediction time is uncertain.",
    "Client names, URLs, and private queries": "Not used to protect privacy.",
    "client_hash_id and content_hash_id": "Not used as features because they identify records."
}

print("Excluded fields and reasons:")
for field, reason in excluded_fields.items():
    print(f"- {field}: {reason}")

print("\nFeatures included:")
print(list(X.columns))

print("\nFeature vector shape:", X.shape)

Excluded fields and reasons:
- Future-period fields: Not used because they may contain information from after prediction time.
- Label-derived columns: Not used because they may reveal the outcome.
- Product flags: Not used because their availability at prediction time is uncertain.
- Client names, URLs, and private queries: Not used to protect privacy.
- client_hash_id and content_hash_id: Not used as features because they identify records.

Features included:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']

Feature vector shape: (10000, 3)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.